# schema equivalences

$metaschema(url)(schema)(input)=object$

In [1]:
from nbref import htmls

from nbref import meta, Schema, schema, htmls
from nbref.utils import iter_values
import pandas

In [2]:
s = Schema(properties=dict(foo={
    "if": dict(
        type="integer"
    ),
    "then": dict(
        minimum=0
    ),
    "else": dict(
        type="string"
    )
}))

In [5]:
s(foo=True)["foo"].expand().schema["not"]()

true

In [ ]:
x = Schema.Object.dispatch(1).modify(**{
    "if": dict(
        type="integer"
    ),
    "then": dict(
        minimum=0
    ),
    "else": dict(
        type="string"
    )
})

In [ ]:
x.schema.expand(1)

In [3]:
x

1

In [3]:
x.schema.expand(x)

{
    "if": {
        "type": "integer"
    },
    "then": {
        "minimum": 0
    },
    "else": {
        "type": "string"
    }
}

In [4]:
Schema(type=["number", "string"]).expand()(1.2).pipe(htmls.Html().render)

AttributeError: 'Html' object has no attribute 'render_not'

In [ ]:
htmls.

In [2]:
nb_schema = Schema.from_file(schema.parent / "nb.yaml").expand()

In [3]:
%%
    # include @index when unrolling listens and dicts, good convention
    nb_schema_table = Schema(nb_schema, properties=
```yaml
cells:
    items:
        properties:
            id:
                writeOnly: true
                "@type": "@id"
```
    ).expand().set_root(nb_schema)

# include @index when unrolling listens and dicts, good convention
nb_schema_table = Schema(nb_schema, properties=
 
 cells : 
 items : 
 properties : 
 cell : 
 role : link 
 "@type" : "@id" 
 id : 
 writeOnly : true 
 "@type" : "@id" 
 
 
 ).expand().set_root(nb_schema)

    nb = Schema.Object.from_file(
        "/Users/tonyfast/antisocial/postmodern.ipynb"
    ).set_schema(nb_schema_table)

In [4]:
    nb = Schema.Object.from_file(
        "/Users/tonyfast/tonyfast/tonyfast/xxiv/2024-01-20-colgroup-schema.ipynb"
    ).set_schema(nb_schema_table).set_name("main")
    for index, cell in enumerate(nb["cells"], 1):
        if cell["cell_type"] == "markdown":
            cell["outputs"] = [dict(
                output_type="display_data",
                data={
                    "text/markdown": cell["source"]
                }, metadata={}
            )]
        # cell["cell"] = index 

In [5]:
    nb.pipes(htmls.Html().render, str, HTML)

11 text/markdown
md
dialog! false
11 text/plain
11 text/markdown
md
11 text/plain
dialog! false
11 text/markdown
md
11 text/plain
dialog! false
dialog! false


In [8]:
nb.set_name("yo").pipes(htmls.Html().render, str, extruct.extract, get("microdata"))

[{'type': 'https://deathbeds.github.io/nbref/schema/nb.yaml',
  'properties': {'dcterms:description': '',
   '#': {'properties': {'0': {'properties': {'metadata': {'value': ''}}},
     '1': {'properties': {'execution_count': '22',
       'metadata': {'value': ''},
       '0': {'properties': {'data': {'value': "text/plain\n\n{'allOf': [{'$ref': '#/$defs/A'}, {'$ref': '#/$defs/B'}],\\n\n'$defs': {'A': {'properties': {'foo': {}, 'bar': {}}},\\n\n'B': {'properties': {'baz': {}}}}}"},
         'execution_count': '22',
         'metadata': {'value': ''}}},
       '1': {'properties': {'data': {'value': 'text/markdown\n\nmake a schema with subschema\\n\n\\n\n(schema := tomli.loads(\\"\\".join(\\n\n\\n\n```toml\\n\n[[allOf]]\\n\n\\"$ref\\" = \\"#/$defs/A\\"\\n\n[[allOf]]\\n\n\\"$ref\\" = \\"#/$defs/B\\"\\n\n[\\"$defs\\".A.properties.foo]\\n\n[\\"$defs\\".A.properties.bar]\\n\n[\\"$defs\\".B.properties.baz]\\n\n```\\n\n\\n\n.splitlines(1)[1:-1])))\n\ntext/plain\n\n<IPython.core.display.Markdown 

In [6]:
import extruct

In [6]:
htmls.Html()

Html(options=Options(renderer=...), role_mapping={'region': <function html_region at 0x109f199b0>}, format_mapping={}, content_mapping={})

In [1]:
from nbref import htmls

In [15]:
    for i, cell in enumerate(nb["cells"]):
        cell["cell"] = i+1
    df = pandas.DataFrame(nb["cells"])

    columns = cell.schema
    required = columns.get("required", [])
    required=required + [
        k for k in df.columns if k not in required
    ]
    indexes = 0
    for k in required:
        if "@id" in cell.schema.property(k).get("@type", ""):
            indexes += 1
            continue
        
    df = df.reindex(required, axis=1)

In [69]:
Schema(
    title="column visibility",
    properties={k: dict(type="boolean", default=True) for k in cell.schema.properties()}
)

{
    "title": "column visibility",
    "properties": {
        "cell": {
            "type": "boolean",
            "default": true
        },
        "id": {
            "type": "boolean",
            "default": true
        },
        "cell_type": {
            "type": "boolean",
            "default": true
        },
        "execution_count": {
            "type": "boolean",
            "default": true
        },
        "source": {
            "type": "boolean",
            "default": true
        },
        "metadata": {
            "type": "boolean",
            "default": true
        },
        "outputs": {
            "type": "boolean",
            "default": true
        }
    }
}

In [70]:
    from nbref.utils import el

In [71]:
    table = el("table", el("thead>tr"), el("tbody"))
    for i, column in enumerate(df.columns):
        tags = nb["cells"][0].schema.property(column).types()
        el(table.thead.tr, el("th", column, scope="col", klass=tags))
        
    for i, row in df.iterrows():
        tr = el(table, el("tr"))
        for j, column in enumerate(required):
            tags = nb["cells"][0].schema.property(column).types()
            tags += []
            aria = dict(rowcount=len(df), rowindex=i+1)
            cell = el("th" if j < indexes else "td", klass=tags)
            el(tr, cell)
            value = row[column]
            el(cell, str(value))
            

    HTML(str(table))

cell,id,execution_count,cell_type,source,outputs,metadata
1,8c80c42a-e0a0-4d02-abb0-fe0c032f6247,nan,markdown,"['# semantic representation of columns based on schema\n', '\n', 'schema may define the columnar properties of a table.\n', 'that can be canonically presented using the `colgroup` and `col` elements\n', 'that allow for bulk operations on column styles.']",nan,{}
2,d3ac8daa-05db-4876-8f4e-9daeb6ca4918,22.0,code,"['make a schema with subschema\n', '\n', ' (schema := tomli.loads("""".join(\n', '\n', '```toml\n', '[[allOf]]\n', '""$ref"" = ""#/$defs/A""\n', '[[allOf]]\n', '""$ref"" = ""#/$defs/B""\n', '[""$defs"".A.properties.foo]\n', '[""$defs"".A.properties.bar]\n', '[""$defs"".B.properties.baz]\n', '```\n', '\n', ' .splitlines(1)[1:-1])))']","[{'data': {'text/plain': [""{'allOf': [{'$ref': '#/$defs/A'}, {'$ref': '#/$defs/B'}],\n"", "" '$defs': {'A': {'properties': {'foo': {}, 'bar': {}}},\n"", "" 'B': {'properties': {'baz': {}}}}}""]}, 'execution_count': 22, 'metadata': {}, 'output_type': 'execute_result'}, {'data': {'text/markdown': ['make a schema with subschema\n', '\n', ' (schema := tomli.loads("""".join(\n', '\n', '```toml\n', '[[allOf]]\n', '""$ref"" = ""#/$defs/A""\n', '[[allOf]]\n', '""$ref"" = ""#/$defs/B""\n', '[""$defs"".A.properties.foo]\n', '[""$defs"".A.properties.bar]\n', '[""$defs"".B.properties.baz]\n', '```\n', '\n', ' .splitlines(1)[1:-1])))'], 'text/plain': ['<IPython.core.display.Markdown object>']}, 'metadata': {}, 'output_type': 'display_data'}]",{}
3,5fd1e708-13db-4d4f-aa08-7cdccf5ab511,23.0,code,"['the form we create uses `colgroup` and `col` elements as demonstrated next\n', '\n', '```html\n', '{{table.prettify()}}\n', '```\n', '\n', 'and it is generated directly from schema using the following code generate example `colgroup` and `col` elements\n', '\n', ' import bs4, jsonpointer\n', ' soup = bs4.BeautifulSoup(features=""html5lib"")\n', ' soup.append(table := soup.new_tag(""table""))\n', ' table.append(colgroup := soup.new_tag(""colgroup""))\n', ' table.append(props := soup.new_tag(""colgroup""))\n', ' for s in schema[""allOf""]:\n', ' id = jsonpointer.JsonPointer.from_parts((ref:=s.get(""$ref"")).split(""/"")[1:])\n', ' colgroup.append(col := soup.new_tag(""col""))\n', ' p = id.get(schema).get(""properties"")\n', ' col.attrs.update(id=str(id), span=len(p))\n', ' for k in p:\n', ' props.append(x := soup.new_tag(""col""))\n', ' x.attrs.update(id=jsonpointer.JsonPointer.from_parts(ref.split(""/"")[1:] + [""properties"", k]))\n', '\n']","[{'data': {'text/markdown': ['the form we create uses `colgroup` and `col` elements as demonstrated next\n', '\n', '```html\n', '<table>\n', ' <colgroup>\n', ' <col id=""/$defs/A"" span=""2""/>\n', ' <col id=""/$defs/B"" span=""1""/>\n', ' </colgroup>\n', ' <colgroup>\n', ' <col id=""/$defs/A/properties/foo""/>\n', ' <col id=""/$defs/A/properties/bar""/>\n', ' <col id=""/$defs/B/properties/baz""/>\n', ' </colgroup>\n', '</table>\n', '\n', '```\n', '\n', 'and it is generated directly from schema using the following code generate example `colgroup` and `col` elements\n', '\n', ' import bs4, jsonpointer\n', ' soup = bs4.BeautifulSoup(features=""html5lib"")\n', ' soup.append(table := soup.new_tag(""table""))\n', ' table.append(colgroup := soup.new_tag(""colgroup""))\n', ' table.append(props := soup.new_tag(""colgroup""))\n', ' for s in schema[""allOf""]:\n', ' id = jsonpointer.JsonPointer.from_parts((ref:=s.get(""$ref"")).split(""/"")[1:])\n', ' colgroup.append(col := soup.new_tag(""col""))\n', ' p = id.get(schema).get(""properties"")\n', ' col.attrs.update(id=str(id), span=len(p))\n', ' for k in p:\n', ' props.append(x := soup.new_tag(""col""))\n', ' x.attrs.update(id=jsonpointer.JsonPointer.from_parts(ref.split(""/"")[1:] + [""properties"", k]))\n'], 'text/plain': ['<IPython.core.display.Markdown object>']}, 'metadata': {}, 'output_type': 'display_data'}]",{}


In [27]:
    df.columns.union(columns.properties())

Index(['cell', 'cell_type', 'execution_count', 'id', 'metadata', 'outputs',
       'source'],
      dtype='str')

In [ ]:
    df.c

In [22]:
    class TableOptions:
        columns_schema: Schema = field(default_factory=Schema)

In [ ]:
    for i, row in object.iterrows():
        

In [16]:
cell.schema

{
    "properties": {
        "cell": {
            "type": "integer"
        },
        "id": {
            "type": "string",
            "class": [
                "hidden"
            ],
            "@type": "@id"
        },
        "cell_type": {
            "type": "string",
            "enum": [
                "code",
                "markdown",
                "raw",
                "file"
            ]
        },
        "execution_count": {
            "type": "integer",
            "if": {
                "type": null
            },
            "then": {
                "default": -1
            }
        },
        "source": {
            "type": "string",
            "format": "textarea",
            "role": "details",
            "style": {
                "width": "100%"
            }
        },
        "metadata": {
            "role": "dialog",
            "$ref": "#/$defs/metadata"
        },
        "outputs": {
            "type": "array",
            "readOnly": tr

In [29]:
columns.required()

[
    "cell",
    "id",
    "execution_count",
    "cell_type",
    "source",
    "outputs"
]

In [21]:
    table_nb["properties"]["cells"]["items"]

{
    "properties": {
        "id": {
            "type": "string",
            "class": [
                "hidden"
            ],
            "@type": "@id"
        },
        "cell": {
            "type": "integer"
        },
        "cell_type": {
            "type": "string",
            "enum": [
                "code",
                "markdown",
                "raw",
                "file"
            ]
        },
        "execution_count": {
            "type": "integer",
            "if": {
                "type": null
            },
            "then": {
                "default": -1
            }
        },
        "source": {
            "type": "string",
            "format": "textarea",
            "role": "details",
            "style": {
                "width": "100%"
            }
        },
        "metadata": {
            "role": "dialog",
            "$ref": "#/$defs/metadata"
        },
        "outputs": {
            "type": "array",
            "readOnly": tr

In [8]:
Schema(nb.property("cells")["items"], role="table", properties=dict(
    id=
))

{
    "required": [
        "cell",
        "id",
        "execution_count",
        "cell_type",
        "source",
        "outputs"
    ],
    "properties": {
        "cell": {
            "type": "integer"
        },
        "id": {
            "type": "string",
            "class": [
                "hidden"
            ],
            "@type": "@id"
        },
        "cell_type": {
            "type": "string",
            "enum": [
                "code",
                "markdown",
                "raw",
                "file"
            ]
        },
        "execution_count": {
            "type": "integer",
            "if": {
                "type": null
            },
            "then": {
                "default": -1
            }
        },
        "source": {
            "type": "string",
            "format": "textarea",
            "role": "details",
            "style": {
                "width": "100%"
            }
        },
        "metadata": {
            "role

In [7]:
%%
```yaml

```

In [ ]:
def table_from_rows(schema, object, options=None, **attrs):
    pass

def table_from_elements(schema, object, options=None, **attrs):
    pass



In [3]:
meta.maps[15]

{}

In [6]:
count(meta.maps)

45

In [4]:
meta["properties"]

{}

s = Schema.from_file(schema.parent / "meta.yaml")

In [7]:
import referencing

In [8]:
resource = referencing.Resource(s.builtin(), referencing.jsonschema.DRAFT202012)

In [6]:
s.Ref(s.builtin()["properties"]["parent"]["$dynamicRef"]).set_root(
    s
).resolve()

{
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "$id": "https://json-schema.org/draft/2020-12/schema",
    "$vocabulary": {
        "https://json-schema.org/draft/2020-12/vocab/core": true,
        "https://json-schema.org/draft/2020-12/vocab/applicator": true,
        "https://json-schema.org/draft/2020-12/vocab/unevaluated": true,
        "https://json-schema.org/draft/2020-12/vocab/validation": true,
        "https://json-schema.org/draft/2020-12/vocab/meta-data": true,
        "https://json-schema.org/draft/2020-12/vocab/format-annotation": true,
        "https://json-schema.org/draft/2020-12/vocab/content": true
    },
    "$dynamicAnchor": "meta",
    "title": "Core and Validation specifications meta-schema",
    "allOf": [
        {
            "$ref": "meta/core"
        },
        {
            "$ref": "meta/applicator"
        },
        {
            "$ref": "meta/unevaluated"
        },
        {
            "$ref": "meta/validation"
        },
        

In [2]:
nb = Schema.from_file(schema.parent / "nb.yaml").expand()

In [3]:
document = Schema.Object.from_file("refs.ipynb").pipe(nb).modify(
    properties=dict(
        cells=dict(maxItems=2)
    )
)
document["cells"] = document["cells"][:10]
for i, cell in enumerate(document["cells"], 1):
    cell["cell"] = i
    execution_count = cell.get("execution_count", -1)
    if execution_count is None:
        pass
    elif execution_count == -1:
        cell["execution_count"] = execution_count
        
    if cell["cell_type"] == "markdown":
        cell["outputs"] = [dict(data={"text/markdown": cell["source"]})]
    
document.display()

cell
execution_count
execution_count
cell
version


In [4]:
document.schema.property("cells")

{
    "maxItems": 2,
    "type": "array",
    "items": {
        "required": [
            "cell",
            "id",
            "execution_count",
            "cell_type",
            "source",
            "outputs"
        ],
        "properties": {
            "cell": {
                "type": "integer"
            },
            "id": {
                "type": "string",
                "class": [
                    "hidden"
                ],
                "@type": "@id"
            },
            "cell_type": {
                "type": "string",
                "enum": [
                    "code",
                    "markdown",
                    "raw",
                    "file"
                ]
            },
            "execution_count": {
                "type": "integer",
                "if": {
                    "type": null
                },
                "then": {
                    "default": -1
                }
            },
            "source": {
       

In [18]:
document.schema["maxLength"]

KeyError: 'maxLength'

In [4]:
Schema(type=["integer", "string"]).expand(1)["type"]

integer

In [5]:
x = Schema(type="integer", oneOf=[
    dict(minimum=0, maximum=10),
    dict(minimum=100, maximum=200),
    dict(minimum=300, maximum=500),
])(11)

In [6]:
x.schema.expand(x)

{
    "type": "integer",
    "oneOf": [
        {
            "minimum": 0,
            "maximum": 10
        },
        {
            "minimum": 100,
            "maximum": 200
        },
        {
            "minimum": 300,
            "maximum": 500
        }
    ],
    "not": {
        "default": 11
    }
}

In [7]:
    schema = Schema.from_string("""
    type: object
    maxExamples: 3
    maxProperties: 3
    additionalProperties: 
        type: integer
    """)

In [8]:
%%
    @schema.given()
    def test_schema(object):
        assert object.schema, \
ensure the schema is maintained through property access.
        
        for object in iter_values(object):
the novel feature of these objects is that the schema can be tracked
as the object is accessed. the objects 

            assert object.schema and object.schema["type"],\
for the specific test cases must be defined to always know the type.
    
    test_schema() or True

True

@schema.given()
def test_schema(object):
 assert object.schema, \
 
 ensure the schema is maintained through property access. 
 for object in iter_values(object):
 
 the novel feature of these objects is that the schema can be tracked
as the object is accessed. the objects 
 assert object.schema and object.schema["type"],\
 
 for the specific test cases must be defined to always know the type. 
 test_schema() or True

In [8]:
%%
the following tests that validate metaschema tracking are more costly
because of frequent dereferencing. 
the meta schema is handy for validating schema, but using in representation
can prove costly until those inefficiencies are researched and addressed.
there is still value in tracking the space


    @meta(schema).given()
    def test_metaschema(object):
        assert object.schema and object.schema.schema,\
ensure the schema and metaschema are maintained through property access.
        
        for object in iter_values(object):
            assert object.schema and object.schema.schema
            assert (type := object.schema["type"]) and type.schema
    test_metaschema() or True

True

the following tests that validate metaschema tracking are more costly
because of frequent dereferencing.
the meta schema is handy for validating schema, but using in representation
can prove costly until those inefficiencies are researched and addressed.
there is still value in tracking the space 
 @meta(schema).given()
def test_metaschema(object):
 assert object.schema and object.schema.schema,\
 
 ensure the schema and metaschema are maintained through property access. 
 for object in iter_values(object):
 assert object.schema and object.schema.schema
 assert (type := object.schema["type"]) and type.schema
test_metaschema() or True

In [22]:
%%
## the one of equivalence

```yaml
type: integer
```

```yaml
oneOf:
- type: integer
```

any schema or subschema definition can be wrapped in a oneOf with one subschema.

the one of equivalence 
 type : integer 
 
 
 oneOf : 
 - type : integer 
 
 
 any schema or subschema definition can be wrapped in a oneOf with one subschema.

In [23]:
%%
## the all of equivalence

```yaml
type: integer
minimum: 0
maximum: 100
```

```yaml
allOf:
- type: integer
- minimum: 0
- maximum: 100
```



it is possible separate any schema into an all of schema. 
this expansion makes sense if schema define two type spaces sim

the all of equivalence 
 type : integer 
 minimum : 0 
 maximum : 100 
 
 
 allOf : 
 - type : integer 
 - minimum : 0 
 - maximum : 100 
 
 
 it is possible separate any schema into an all of schema.
this expansion makes sense if schema define two type spaces sim

In [24]:
%%
## the one of type equivalence

```yaml
type: [array, object]
```

```yaml
oneOf:
- type: array
- type: object
```

the one of type equivalence 
 types =\
 
 type : [ array , object ] 
 
 
 types_expanded =\
 
 oneOf : 
 - type : array 
 - type : object

## the not equivalence

the not equivalence describes the empty/null space. bad examples may be defined and the `not` can be added to describes inconsistent inputs.

```yaml
type: boolean
default: true
examples:
- "true"
- 1
```

```yaml
type: boolean
default: true
examples:
- "true"
- 1
not:
    type: [string, number]
```

In [ ]:
## anyof

```yaml
type: []